# Exp 5: all-fish position-map raster

This is the maintained all-fish raster workflow for `Exp_5_map_positions`. It uses the shared `src` helpers to load the validated Exp 5 cohort, build z-score rasters and mean traces, and inspect left/right activity. The experiment-specific cohort, timing, and stimulus order are defined in the settings cell. Sections 4a/4b add the Exp 8-style independent stimulus panels, including reuse of this notebook's exact Cell 06 neuron orders.

In [ ]:
# Cell 01 - Setup
from pathlib import Path
import sys
import importlib
from IPython.display import display

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

repo_root = next(path for path in [Path.cwd(), *Path.cwd().parents] if (path / 'src').is_dir())
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

import src.active_neuron_analysis as active_analysis
import src.data_loading as exio
import src.multifish_matrices as multifish_matrices
import src.plotting_common as plot_common
import src.plotting_diagnostics as plot_diagnostics
import src.several_fish_diagnostics as several_diagnostics
import src.several_fish_figures as several_figures
import src.several_fish_loading as several_loading
import src.several_fish_reporting as several_reporting
import src.trial_alignment as trial_alignment
# Refresh shared sorters and panel preparation before the figure consumers.
for module_name in ('src.neuron_ordering', 'src.plotting_all_fish',
                    'src.several_fish_diagnostics'):
    importlib.reload(importlib.import_module(module_name))
for module in (plot_common, plot_diagnostics, several_figures, several_loading, several_reporting):
    importlib.reload(module)


### Save figures independently as SVG

Each plotting cell has its own `SAVE_..._SVG` switch, defaulting to `False`.
Set a cell's switch to `True` and rerun that cell to save all its figures.
`SVG_OUTPUT_DIR` resolves the local Lab folder and writes to
`Results/Paper/figures/main/fig04/assets/generated`. Filenames identify the
cell, plot and side/layout; rerunning an enabled export updates those files.
SVG text stays editable; raster heatmaps remain embedded images.


In [ ]:
# Cell 02 - Exp 5 settings
# Cohort, stimulus IDs, and timing were taken from Exp5_map_positions_several.ipynb.
USER_SETTINGS = {
    'experiment_name': 'Exp_5_map_positions',
    'analysis_label': 'Exp 5 map positions',
    'lab_root': exio.resolve_lab_root(),  # Discover the local Lab folder on either computer.
    # Maintained cohort with complete significant-raster and kept-neuron caches.
    'fish_ids': ['L684_f01', 'L684_f07', 'L733_f01', 'L733_f06', 'L733_f07'],
    'selected_blocks': [f'B{number}' for number in range(1, 5)],
    'verbose_loading': False,
    'timing': {'fps_2p': 2.0, 't_pre_s': 5.0, 't_post_s': 25.0, 'motion_onset_s': 8.0, 'tau_s': 6.0},
    # Left controls/positions/flicker/rocking, followed by the matched right stimuli.
    'stim_order': [7, 3, 4, 5, 6, 1, 13, 12, 8, 9, 10, 11, 2, 14],
    'combine_mode': 'mean',
    'trace_type': 'zscore',
    'raster_sort_mode': 'left_right_index',
    'left_control': 7,   # LeBcontrol
    'right_control': 12, # RiBcontrol
}

# Shared destination; each plotting cell independently decides whether to save.
SVG_OUTPUT_DIR = USER_SETTINGS['lab_root'] / (
    'Results/Paper/figures/main/fig04/assets/generated'
)

REPORT_SETTINGS = {
    'save_report': False,
    'run_label': 'exp5_map_positions_all_fish_raster',
    'report_format': 'html',
    'export_notebook': True,
    'fallback_to_html': True,
    'comments': 'Enable only after reviewing the Exp 5 figures.',
}


In [ ]:
# Cell 03 - Load the selected Exp 5 cohort and validate every configured stimulus ID.
raster_inputs = several_loading.load_and_preflight_fish_raster_inputs(USER_SETTINGS)
globals().update(raster_inputs)
stimuli_colors, stimuli_linestyles = plot_common.build_stimulus_style_maps(stimuli_names=stimuli_names)
print('Detected Exp 5 stimulus map:', stimuli_id_map)


In [ ]:
# Cell 04 - All-fish z-score raster.
SAVE_CELL04_SVG = False  # True saves every figure from this cell as SVG.
FIGSIZE = (13, 8)
raster_result = several_figures.build_all_fish_raster_figure(
    all_fish_data, fish_ids, stim_order, reference_fish, timing, settings,
    stimuli_colors, stimuli_linestyles, figsize=FIGSIZE,
)
flat_matrix_all_fish = raster_result['flat_matrix_all_fish']
if SAVE_CELL04_SVG:
    plot_common.save_publication_figure(
        raster_result['figure'], SVG_OUTPUT_DIR / 'exp05_cell04_all_fish_zscore', formats=('svg',), dpi=600,
    )
plt.show()


## 4a. Shared activity decisions and Cell 06 neuron orders

Prepare the left/right activity decisions and exact Cell 06 neuron orders once.
The activity settings match the original Exp 5 Cell 06 defaults. Neurons are
grouped by their active/inactive pattern in stimulus order, then ordered within
each group by mean significant-raster activity. Section 4b and Cell 06 reuse
these same cached orders. Rerun after changing fish, stimulus order, timing,
or activity rules.


In [ ]:
# Prepare activity decisions and both Cell 06 orders once.
SECTION6_ACTIVE_SETTINGS = {
    'active_fraction_threshold': 0.10, 'min_epoch_s': 1.0,
    'min_active_reps': 2, 'expected_reps': 4, 'require_expected_reps': True,
}
# Exp 5 side membership comes from its validated map; preserve configured order.
RASTER_STIMULI_BY_SIDE = {
    'left': [stimuli_id_map[name] for name in
             ('LeBcontrol', 'LeB1', 'LeB2', 'LeB3', 'LeB4', 'FlickL', 'RockL')],
    'right': [stimuli_id_map[name] for name in
              ('RiBcontrol', 'RiB1', 'RiB2', 'RiB3', 'RiB4', 'FlickR', 'RockR')],
}
OVERLAP_STIMULI = {
    side: [sid for sid in stim_order if sid in ids]
    for side, ids in RASTER_STIMULI_BY_SIDE.items()
}
overlap_results = several_diagnostics.build_left_right_active_overlap_diagnostics(
    all_fish_data, fish_ids, reference_fish, timing, OVERLAP_STIMULI,
    motion_onset_s=timing['motion_onset_s'], tau_s=timing['tau_s'],
    **SECTION6_ACTIVE_SETTINGS,
)


## 4b. Independent stimulus panels

The same layouts as Exp 8 section 4b: separate left/right figures for z-score
rasters, mean traces, and rasters above their corresponding mean traces.
Each stimulus has its own time axis and averages the four Exp 5 repetitions.
Every raster retains all kept neurons. Thin colored curves show individual
fish; black shows the equally weighted fish mean. Phase shading and markers
use Exp 5's loaded timing.

Run the preparation cell once per sorting/data change. `PANEL_SORT_MODE` accepts
`cell06` (default), `corravg`, `rastermap`, `left_right_index`, or `unsorted`.
`cell06` reuses section 4a's exact, independent left/right orders, also used by
Cell 06 below. Other modes share one order across sides. Rastermap needs its
optional package (`%pip install "rastermap>=1.0,<2"`, then restart the kernel).
Sorting only changes row order and preserves the original z-scores.

For appearance changes, rerun only the following plotting cell. Choose
`PANEL_SIDE`, and edit `PANEL_STIMULI_BY_SIDE` to select IDs in your preferred
column order. Font (6 pt by default), figure dimensions, colormap, z-score
limits, and mean-trace limits stay beside the plotting call. Appearance changes
reuse the cached sorting; rerun preparation if you add newly loaded stimuli.


In [ ]:
# PREPARE / SORT — rerun only for sorting or data changes.
PANEL_SORT_MODE = 'cell06'# 'rastermap'#'cell06'  # 'cell06', 'corravg', 'rastermap', 'left_right_index', 'unsorted'.
PANEL_REUSE_RASTER_ORDER = False  # Applies to the other sorting modes.
panel_order_sources = overlap_results if PANEL_SORT_MODE == 'cell06' else None
panel_base_mode = 'unsorted' if PANEL_SORT_MODE == 'cell06' else PANEL_SORT_MODE
panel_order = (raster_result['neuron_order'] if PANEL_REUSE_RASTER_ORDER
               and panel_base_mode == settings['raster_sort_mode'] else None)
if PANEL_SORT_MODE == 'cell06':
    panel_order = np.arange(len(overlap_results['left']['diagnostic_neuron_order']))
stimulus_panel_display = several_diagnostics.build_stimulus_panel_display(
    all_fish_data, fish_ids, stim_order, reference_fish, timing,
    {**settings, 'raster_sort_mode': panel_base_mode}, neuron_order=panel_order,
)


In [ ]:
# APPEARANCE ONLY - rerun freely; this cell never computes the sorting.
SAVE_PANEL_SVG = True # True saves every figure from this cell as SVG.
MM = 1 / 25.4
PANEL_SIDE = 'both'  # 'both', 'left', or 'right'; each side gets separate figures.
PANEL_STIMULI_BY_SIDE = {side: list(ids) for side, ids in OVERLAP_STIMULI.items()}
# To show a subset or reorder columns, replace the line above, for example:
# PANEL_STIMULI_BY_SIDE = {'left': [7, 3, 1, 13], 'right': [12, 8, 2, 14]}
PANEL_FONT_PT = 6
PANEL_RASTER_FIGSIZE = (150 * MM, 45 * MM)
PANEL_MEAN_FIGSIZE = (150 * MM, 42.5 * MM)
PANEL_COMBINED_FIGSIZE = (150 * MM, 65 * MM)
PANEL_Z_LIMITS = (0,6)#raster_result['image'].get_clim()  # Or a tuple, e.g. (0, 10).
PANEL_MEAN_YLIM = None  # Automatic shared range, or a tuple, e.g. (-1, 5).
PANEL_CMAP = 'Greys'
PANEL_STYLE = {
    'font.family': 'Arial', 'font.size': PANEL_FONT_PT,
    'axes.labelsize': PANEL_FONT_PT, 'axes.titlesize': PANEL_FONT_PT,
    'figure.titlesize': PANEL_FONT_PT, 'xtick.labelsize': PANEL_FONT_PT,
    'ytick.labelsize': PANEL_FONT_PT, 'legend.fontsize': PANEL_FONT_PT,
    'legend.title_fontsize': PANEL_FONT_PT, 'axes.linewidth': 0.6,
    'lines.linewidth': 1.0, 'lines.markersize': 3,
    'xtick.major.width': 0.6, 'ytick.major.width': 0.6,
    'xtick.major.size': 2.5, 'ytick.major.size': 2.5,
    'figure.dpi': 150, 'savefig.dpi': 600, 'savefig.bbox': None,
    'pdf.fonttype': 42, 'ps.fonttype': 42, 'svg.fonttype': 'none',
    'mathtext.fontset': 'dejavusans', 'figure.facecolor': 'white',
}
if PANEL_SIDE not in ('both', 'left', 'right'):
    raise ValueError("PANEL_SIDE must be 'both', 'left', or 'right'.")
panel_sides = {side: PANEL_STIMULI_BY_SIDE[side]
               for side in ('left', 'right') if PANEL_SIDE in ('both', side)}
with plt.rc_context(PANEL_STYLE):
    stimulus_panel_result = several_figures.plot_all_fish_stimulus_figures(
        stimulus_panel_display, panel_sides, side_order_sources=panel_order_sources,
        z_limits=PANEL_Z_LIMITS, mean_ylim=PANEL_MEAN_YLIM,
        raster_figsize=PANEL_RASTER_FIGSIZE, mean_figsize=PANEL_MEAN_FIGSIZE,
        combined_figsize=PANEL_COMBINED_FIGSIZE, cmap=PANEL_CMAP,
    )
    for panel_name, figure in stimulus_panel_result['figures'].items():
        if SAVE_PANEL_SVG:
            plot_common.save_publication_figure(
                figure, SVG_OUTPUT_DIR / f'exp05_cell4b_{PANEL_SORT_MODE}_{panel_name}', formats=('svg',), dpi=600,
            )
        display(figure)
        plt.close(figure)


In [ ]:
# Cell 05 - Mean z-score traces in the configured Exp 5 stimulus order.
SAVE_CELL05_SVG = False  # True saves every figure from this cell as SVG.
FIGSIZE = (12, 4)
mean_trace_result = several_figures.build_plot_all_fish_mean_zscore_traces(
    all_fish_data, fish_ids, stim_order, reference_fish, timing,
    stimuli_colors, stimuli_linestyles, figsize=FIGSIZE,
)
if SAVE_CELL05_SVG:
    plot_common.save_publication_figure(
        mean_trace_result['figure'], SVG_OUTPUT_DIR / 'exp05_cell05_mean_zscore', formats=('svg',), dpi=600,
    )


In [ ]:
# Cell 06 - Left/right active-neuron overlap and significant-raster diagnostics.
SAVE_CELL06_SVG = False  # True saves every figure from this cell as SVG.
OVERLAP_FIGSIZE = (5, 5)
RASTER_FIGSIZE = (16, 8)
# Reuse section 4a decisions/orders, exactly as section 4b does.
overlap_plots = several_figures.plot_prepared_left_right_active_overlap_diagnostics(
    overlap_results, reference_fish, timing,
    show_overlap=True, show_significant_raster=False, overlap_figsize=OVERLAP_FIGSIZE,
    raster_figsize=RASTER_FIGSIZE,
)
if SAVE_CELL06_SVG:
    for side, result in overlap_plots.items():
        plot_common.save_publication_figure(
            result['overlap_figure'], SVG_OUTPUT_DIR / f'exp05_cell06_{side}_active_overlap', formats=('svg',), dpi=600,
        )
for side in ('left', 'right'):
    diagnostic_data = overlap_results[side]
    diagnostic = diagnostic_data['active_trace_diagnostic']
    if diagnostic['trace_matrix'].shape[0] == 0:
        print(f'No active neurons passed the {side} settings.')
        continue
    diagnostic_figure, _, _ = plot_diagnostics.plot_active_trace_decision_diagnostic(
        diagnostic, fps_2p=timing['fps_2p'], stimuli_durations=reference_fish['stimuli_durations'],
        t_pre_s=timing['t_pre_s'], sort_mode=None,
        neuron_order=diagnostic_data['diagnostic_neuron_order'], trace_cmap='Greys',
        trace_title=f'{side.capitalize()} significant raster', show_active_count_trace=True,
        active_count_threshold=0.5, active_count_color='black', active_count_ylabel='# Active neurons',
        active_count_height_ratio=2.4, figsize=RASTER_FIGSIZE,
    )
    if SAVE_CELL06_SVG:
        plot_common.save_publication_figure(
            diagnostic_figure, SVG_OUTPUT_DIR / f'exp05_cell06_{side}_significant_raster', formats=('svg',), dpi=600,
        )
    plt.show()


## 6c. Combined population and shared-neuron summary

The same three-row layout as Exp 8 section 6c, with one figure per side:
population activity, recruitment, and response strength in pairwise shared
neurons. Sage dots show fish; slate blue shows the equally weighted fish mean.
Only the bottom row has stimulus labels; Control's self-comparison stays n/a.

All rows use each Exp 5 stimulus's loaded motion onset through motion end plus
5 s, with section 4a's activity rules. Population activity (signed normalized
AUC) and recruitment use all retained neurons. The last row measures
Control minus stimulus normalized AUC within each pair's shared-active neurons,
selected in that same window. No static baseline is subtracted. These are
descriptive measurements using the same trials for selection and measurement.
Exp 5's configured stimulus order is preserved, including flicker before rocking.

Run the next two preparation cells once after loading and section 4a; rerun
them when changing fish, stimuli, timing or activity rules. Figure appearance
and SVG saving are controlled independently in the following plotting cell.


In [ ]:
# Prepare shared-neuron AUC in the same per-stimulus window as the summary.
SHARED_AUC_DECAY_S = 5.0
SUMMARY_TIMING = {**timing, 'motion_onset_s': None, 'tau_s': SHARED_AUC_DECAY_S / 2}
summary_overlap_results = several_diagnostics.build_left_right_active_overlap_diagnostics(
    all_fish_data, fish_ids, reference_fish, SUMMARY_TIMING, OVERLAP_STIMULI,
    motion_onset_s=None, tau_s=SUMMARY_TIMING['tau_s'],
    **SECTION6_ACTIVE_SETTINGS,
)
SHARED_AUC_CONTROLS = {'left': settings['left_control'], 'right': settings['right_control']}
shared_auc_result = several_diagnostics.build_shared_control_auc_analysis(
    all_fish_data, fish_ids, summary_overlap_results, SHARED_AUC_CONTROLS, SUMMARY_TIMING,
    decay_s=SHARED_AUC_DECAY_S,
)


In [ ]:
# Prepare population activity/recruitment once, using Exp 5's own conditions.
# The shared helper's historical 'rocking' group includes non-control motion
# conditions (positions and rocking). Only its activity/recruitment rows are used.
SUMMARY_SIDE_GROUPS = {
    'left': {'control': 'LeBcontrol',
             'rocking': ['LeB1', 'LeB2', 'LeB3', 'LeB4', 'RockL'], 'flicker': ['FlickL']},
    'right': {'control': 'RiBcontrol',
              'rocking': ['RiB1', 'RiB2', 'RiB3', 'RiB4', 'RockR'], 'flicker': ['FlickR']},
}
SUMMARY_CONDITION_LABELS = ['Control', 'Pos 1', 'Pos 2', 'Pos 3', 'Pos 4', 'Rocking', 'Flicker']
combined_summary_analysis = several_diagnostics.build_recruitment_containment_analysis(
    all_fish_data, fish_ids, SUMMARY_SIDE_GROUPS, SUMMARY_TIMING,
    SECTION6_ACTIVE_SETTINGS, SUMMARY_CONDITION_LABELS,
)
# Display order comes from the notebook settings, independently of helper groups.
SUMMARY_STIMULI_BY_SIDE = {side: list(ids) for side, ids in OVERLAP_STIMULI.items()}


Change dimensions, font and colors below without repeating preparation.
Set `SAVE_SUMMARY_SVG=True` to save both summary figures to the shared Fig. 4
`SVG_OUTPUT_DIR`. It is independent of every other cell's save switch.


In [ ]:
# Appearance and optional SVG export only.
SAVE_SUMMARY_SVG = False  # True saves both sides to Fig. 4 assets/generated.
MM = 1 / 25.4
SUMMARY_FIGSIZE = (85 * MM, 110 * MM)  # Per side: width, height.
SUMMARY_FONT_PT = 6
SUMMARY_FISH_COLOR = '#9AAF9F'  # Muted sage.
SUMMARY_MEAN_COLOR = '#385F73'  # Slate blue.
SUMMARY_STYLE = {
    'font.family': 'Arial', 'font.size': SUMMARY_FONT_PT,
    'axes.labelsize': SUMMARY_FONT_PT, 'axes.titlesize': SUMMARY_FONT_PT,
    'figure.titlesize': SUMMARY_FONT_PT, 'xtick.labelsize': SUMMARY_FONT_PT,
    'ytick.labelsize': SUMMARY_FONT_PT, 'legend.fontsize': SUMMARY_FONT_PT,
    'legend.title_fontsize': SUMMARY_FONT_PT, 'axes.linewidth': 0.6,
    'lines.linewidth': 1.0, 'lines.markersize': 3,
    'xtick.major.width': 0.6, 'ytick.major.width': 0.6,
    'xtick.major.size': 2.5, 'ytick.major.size': 2.5,
    'figure.dpi': 150, 'savefig.dpi': 600, 'savefig.bbox': None,
    'pdf.fonttype': 42, 'ps.fonttype': 42, 'svg.fonttype': 'none',
    'mathtext.fontset': 'dejavusans', 'figure.facecolor': 'white',
}
with plt.rc_context(SUMMARY_STYLE):
    combined_summary_figures = several_figures.plot_recruitment_strength_summary(
        combined_summary_analysis, shared_auc_result, figsize=SUMMARY_FIGSIZE,
        fish_color=SUMMARY_FISH_COLOR, mean_color=SUMMARY_MEAN_COLOR,
        side_stimuli=SUMMARY_STIMULI_BY_SIDE,
    )
    for side, (figure, axes) in combined_summary_figures.items():
        if SAVE_SUMMARY_SVG:
            plot_common.save_publication_figure(
                figure, SVG_OUTPUT_DIR / f'exp05_combined_recruitment_strength_{side}',
                formats=('svg',), dpi=600,
            )
        display(figure)
        plt.close(figure)


In [ ]:
# Cell 07 - Reference-active significant rasters, ordered by post-motion onset.
SAVE_CELL07_SVG = False  # True saves every figure from this cell as SVG.
REFERENCE_RASTER_SETTINGS = {
    'left': {'reference_stimulus': 7, 'plot_stimuli': [7, 3, 4, 5, 6, 1, 13]},
    'right': {'reference_stimulus': 12, 'plot_stimuli': [12, 8, 9, 10, 11, 2, 14]},
}
min_active_reps = 0.50

reference_and_plot_stimuli = list(dict.fromkeys(
    [group['reference_stimulus'] for group in REFERENCE_RASTER_SETTINGS.values()]
    + [stimulus for group in REFERENCE_RASTER_SETTINGS.values() for stimulus in group['plot_stimuli']]
))
reference_active_matrices = active_analysis.build_active_neuron_matrices_all_fish(
    all_fish_data=all_fish_data, fish_ids=fish_ids, stim_order=reference_and_plot_stimuli,
    fps_2p=timing['fps_2p'], t_pre_s=timing['t_pre_s'],
    motion_onset_s=timing['motion_onset_s'], tau_s=timing['tau_s'],
    active_fraction_threshold=0.10, min_epoch_s=1.0, min_active_reps=2,
    expected_reps=4, require_expected_reps=True,
)

for side, group in REFERENCE_RASTER_SETTINGS.items():
    selection = trial_alignment.resolve_selected_stimuli(group['plot_stimuli'], reference_fish['stimuli_id_map'],
                                             reference_fish['trial_aligned_traces_raster'].keys())
    reference_selection = trial_alignment.resolve_selected_stimuli([group['reference_stimulus']], reference_fish['stimuli_id_map'],
                                                       reference_fish['trial_aligned_traces_raster'].keys())
    reference_id = reference_selection['stimulus_ids'][0]
    reference_label = reference_selection['stimulus_labels'][0]
    diagnostic = active_analysis.build_pooled_active_trace_diagnostic(
        all_fish_data=all_fish_data, active_matrices=reference_active_matrices, fish_ids=fish_ids,
        stim_order=selection['stimulus_ids'], combine_mode='mean', trial_key='trial_aligned_traces_raster',
    )
    activity_onsets = []
    for fish_id in fish_ids:
        fish = all_fish_data[fish_id]
        reference_key = reference_id if reference_id in fish['trial_aligned_traces_raster'] else str(reference_id)
        reference_raster = np.asarray(fish['trial_aligned_traces_raster'][reference_key], dtype=bool)
        duration = fish['stimuli_durations'][reference_label]
        motion_frame = int(np.clip(round((timing['t_pre_s'] + duration['static_before_sec']) * timing['fps_2p']),
                                   0, reference_raster.shape[1] - 1))
        active_by_frame = reference_raster.mean(axis=2) >= min_active_reps
        post_motion = active_by_frame[:, motion_frame:]
        first_active = np.full(reference_raster.shape[0], np.inf)
        has_active_frame = post_motion.any(axis=1)
        first_active[has_active_frame] = motion_frame + post_motion[has_active_frame].argmax(axis=1)
        activity_onsets.append(first_active)
    activity_onsets = np.concatenate(activity_onsets)
    reference_active = diagnostic['row_metadata'].apply(
        lambda row: bool(reference_active_matrices[row['fish_id']].loc[row['neuron_id'], reference_id]), axis=1
    ).to_numpy()
    keep_rows = np.flatnonzero(reference_active)
    if keep_rows.size == 0:
        print(f'No neurons are strictly active for {side} reference {reference_label}.')
        continue
    selected_diagnostic = {**diagnostic, 'trace_matrix': diagnostic['trace_matrix'][keep_rows],
                           'decision_matrix': diagnostic['decision_matrix'][keep_rows],
                           'row_metadata': diagnostic['row_metadata'].iloc[keep_rows].reset_index(drop=True)}
    onset_order = np.argsort(activity_onsets[keep_rows], kind='stable')
    reference_figure, _, _ = plot_diagnostics.plot_active_trace_decision_diagnostic(
        selected_diagnostic, fps_2p=timing['fps_2p'], stimuli_durations=reference_fish['stimuli_durations'],
        t_pre_s=timing['t_pre_s'], sort_mode=None, neuron_order=onset_order, trace_cmap='Greys',
        trace_title=f'{side.capitalize()} reference-active raster: {reference_label} (≥ {min_active_reps:.0%} repetitions per frame)',
        show_active_count_trace=True, active_count_threshold=0.5, active_count_color='black',
        active_count_ylabel='# Active neurons', active_count_height_ratio=2.4, figsize=RASTER_FIGSIZE,
    )
    if SAVE_CELL07_SVG:
        plot_common.save_publication_figure(
            reference_figure, SVG_OUTPUT_DIR / f'exp05_cell07_{side}_reference_active_raster', formats=('svg',), dpi=600,
        )
    plt.show()


In [ ]:
# Cell 08 - Mean AUC for neurons strictly active in the left position map.
SAVE_CELL08_SVG = False  # True saves every figure from this cell as SVG.
# Edit this list (IDs or names) to choose a different shared-active comparison.
AUC_COMPARISON_STIMULI = [7, 3, 4, 5, 6]
auc_selection = trial_alignment.resolve_selected_stimuli(AUC_COMPARISON_STIMULI, reference_fish['stimuli_id_map'],
                                           reference_fish['trial_aligned_traces_z_core'].keys())
auc_active_matrices = active_analysis.build_active_neuron_matrices_all_fish(
    all_fish_data=all_fish_data, fish_ids=fish_ids, stim_order=auc_selection['stimulus_ids'],
    fps_2p=timing['fps_2p'], t_pre_s=timing['t_pre_s'], motion_onset_s=timing['motion_onset_s'],
    tau_s=timing['tau_s'], active_fraction_threshold=0.10, min_epoch_s=1.0, min_active_reps=2,
    expected_reps=4, require_expected_reps=True,
)
auc_by_fish = []
for fish_id in fish_ids:
    fish = all_fish_data[fish_id]
    shared_active = auc_active_matrices[fish_id].loc[:, auc_selection['stimulus_ids']].all(axis=1).to_numpy()
    if not shared_active.any():
        print(f'{fish_id}: no neurons are strictly active in every selected stimulus.')
        continue
    auc_matrix = multifish_matrices.build_zscore_response_matrix_for_fish(
        fish['trial_aligned_traces_z_core'], auc_selection['stimulus_ids'], fish['stimuli_durations'],
        stimuli_id_map=fish['stimuli_id_map'], kept_neuron_indices=fish['kept_neuron_indices'],
        fps_2p=timing['fps_2p'], t_pre_s=timing['t_pre_s'], motion_onset_s=timing['motion_onset_s'], tau_s=timing['tau_s'],
    )
    auc_by_fish.append(pd.DataFrame({'fish_id': fish_id, 'stimulus': auc_selection['stimulus_labels'],
        'mean_auc': auc_matrix.loc[shared_active, auc_selection['stimulus_labels']].mean(axis=0).to_numpy(),
        'n_shared_active_neurons': int(shared_active.sum())}))
shared_active_auc_summary = pd.concat(auc_by_fish, ignore_index=True) if auc_by_fish else pd.DataFrame()
if shared_active_auc_summary.empty:
    print('No fish contributed shared-active neurons for the selected stimuli.')
else:
    fig, ax = plt.subplots(figsize=(max(6, len(auc_selection['stimulus_labels']) * 1.1), 4.5))
    stimulus_positions = np.arange(len(auc_selection['stimulus_labels']))
    for fish_id, values in shared_active_auc_summary.groupby('fish_id', sort=False):
        y = values.set_index('stimulus').loc[auc_selection['stimulus_labels'], 'mean_auc'].to_numpy()
        ax.plot(stimulus_positions, y, color='0.65', linewidth=1, alpha=0.75)
        ax.scatter(stimulus_positions, y, color='0.25', s=28, alpha=0.9)
    mean_auc = shared_active_auc_summary.groupby('stimulus')['mean_auc'].mean().reindex(auc_selection['stimulus_labels'])
    ax.plot(stimulus_positions, mean_auc, color='black', marker='o', linewidth=2.5, label='Mean across fish')
    ax.set(xticks=stimulus_positions, xticklabels=auc_selection['stimulus_labels'],
           ylabel='Mean z-score AUC per fish', xlabel='Stimulus',
           title='AUC of neurons strictly active in all selected stimuli')
    ax.tick_params(axis='x', rotation=45)
    ax.legend(frameon=False)
    fig.tight_layout()
    if SAVE_CELL08_SVG:
        plot_common.save_publication_figure(
            fig, SVG_OUTPUT_DIR / 'exp05_cell08_shared_active_auc', formats=('svg',), dpi=600,
        )
    plt.show()
    display(shared_active_auc_summary)


In [ ]:
# Cell 09 - Motion-period active neurons per stimulus; each black point is one fish.
SAVE_CELL09_SVG = False  # True saves every figure from this cell as SVG.
FIGSIZE = (8, 3.5)
motion_active_result = several_figures.plot_motion_active_neuron_counts(
    all_fish_data, fish_ids, reference_fish, timing, stim_order, stimuli_colors,
    figsize=FIGSIZE, motion_onset_s=timing['motion_onset_s'], tau_s=timing['tau_s'],
)
active_neuron_counts = motion_active_result['counts']
if SAVE_CELL09_SVG:
    plot_common.save_publication_figure(
        motion_active_result['figure'], SVG_OUTPUT_DIR / 'exp05_cell09_motion_active_counts', formats=('svg',), dpi=600,
    )


In [ ]:
# Cell 10 - Lifetime sparseness and high-sparseness z-score raster.
SAVE_CELL10_SVG = False  # True saves every figure from this cell as SVG.
SPARSENESS_FIGSIZE = (7, 4.5)
RASTER_FIGSIZE = (13, 8)
HIGH_LIFETIME_SPARSENESS = 0.75
sparseness_result = several_figures.plot_lifetime_sparseness_analysis(
    all_fish_data, fish_ids, reference_fish, timing, stim_order, HIGH_LIFETIME_SPARSENESS,
    stimuli_colors, stimuli_linestyles, combine_mode='mean', sparseness_figsize=SPARSENESS_FIGSIZE,
    raster_figsize=RASTER_FIGSIZE, analysis_label=settings['analysis_label'],
)
if SAVE_CELL10_SVG:
    for name in ('sparseness_figure', 'high_sparseness_figure'):
        if sparseness_result[name] is not None:
            plot_common.save_publication_figure(
                sparseness_result[name], SVG_OUTPUT_DIR / f'exp05_cell10_{name}', formats=('svg',), dpi=600,
            )


In [ ]:
# Cell 08 - Recruitment by reference-onset time.
SAVE_REFERENCE_ONSET_SVG = False  # True saves every figure from this cell as SVG.
# This uses the same strict active-neuron decisions as the reference-active raster.
REFERENCE_ONSET_TIME_BIN_S = 0.6  # Editable: width of measured post-motion onset-time bins.
REFERENCE_ONSET_MAX_TIME_S = 9.0  # Editable upper limit; bins span 0 s up to, but not including, this time.
RECRUITMENT_SIDES = ('left',)  # RiB by default; use ('left', 'right') to also plot LeB.

if REFERENCE_ONSET_TIME_BIN_S <= 0:
    raise ValueError('REFERENCE_ONSET_TIME_BIN_S must be positive.')
if REFERENCE_ONSET_MAX_TIME_S <= 0:
    raise ValueError('REFERENCE_ONSET_MAX_TIME_S must be positive.')

reference_onset_recruitment = {}
for side in RECRUITMENT_SIDES:
    if side not in REFERENCE_RASTER_SETTINGS:
        raise KeyError(f'Unknown side {side!r}; choose from {tuple(REFERENCE_RASTER_SETTINGS)}.')
    group = REFERENCE_RASTER_SETTINGS[side]
    selection = trial_alignment.resolve_selected_stimuli(
        group['plot_stimuli'], stimuli_id_map=reference_fish['stimuli_id_map'],
        available_stimuli=reference_fish['trial_aligned_traces_raster'].keys(),
    )
    reference_selection = trial_alignment.resolve_selected_stimuli(
        [group['reference_stimulus']], stimuli_id_map=reference_fish['stimuli_id_map'],
        available_stimuli=reference_fish['trial_aligned_traces_raster'].keys(),
    )
    reference_id = reference_selection['stimulus_ids'][0]
    reference_label = reference_selection['stimulus_labels'][0]

    pooled_rows = []
    for fish_id in fish_ids:
        fish = all_fish_data[fish_id]
        reference_key = reference_id if reference_id in fish['trial_aligned_traces_raster'] else str(reference_id)
        reference_raster = np.asarray(fish['trial_aligned_traces_raster'][reference_key], dtype=bool)
        duration = fish['stimuli_durations'][reference_label]
        motion_frame = int(np.clip(
            round((timing['t_pre_s'] + duration['static_before_sec']) * timing['fps_2p']),
            0, reference_raster.shape[1] - 1,
        ))
        active_by_frame = reference_raster.mean(axis=2) >= min_active_reps
        post_motion = active_by_frame[:, motion_frame:]
        first_active = np.full(reference_raster.shape[0], np.inf)
        has_active_frame = post_motion.any(axis=1)
        first_active[has_active_frame] = motion_frame + post_motion[has_active_frame].argmax(axis=1)

        active_matrix = reference_active_matrices[fish_id]
        reference_column = reference_id if reference_id in active_matrix.columns else str(reference_id)
        for neuron_id in np.flatnonzero(active_matrix[reference_column].to_numpy(dtype=bool)):
            if not np.isfinite(first_active[neuron_id]):
                continue  # A strict reference decision without a measurable post-motion onset.
            row = {
                'fish_id': fish_id, 'neuron_id': neuron_id, 'onset_frame': first_active[neuron_id],
                'onset_time_s': (first_active[neuron_id] - motion_frame) / timing['fps_2p'],
            }
            for stimulus_id, stimulus_label in zip(selection['stimulus_ids'], selection['stimulus_labels']):
                column = stimulus_id if stimulus_id in active_matrix.columns else str(stimulus_id)
                row[stimulus_label] = bool(active_matrix.iloc[neuron_id][column])
            pooled_rows.append(row)

    pooled_rows = pd.DataFrame(pooled_rows)
    if pooled_rows.empty:
        print(f'No reference-active neurons with measurable post-motion onsets for {side} ({reference_label}).')
        continue

    pooled_rows = pooled_rows.loc[
        (pooled_rows['onset_time_s'] >= 0) & (pooled_rows['onset_time_s'] < REFERENCE_ONSET_MAX_TIME_S)
    ].sort_values('onset_time_s', kind='stable').reset_index(drop=True)
    if pooled_rows.empty:
        print(f'No {reference_label}-active neurons have onsets between 0 and {REFERENCE_ONSET_MAX_TIME_S:g} s.')
        continue
    onset_bin_count = int(np.ceil(REFERENCE_ONSET_MAX_TIME_S / REFERENCE_ONSET_TIME_BIN_S))
    pooled_rows['onset_bin'] = np.floor(
        pooled_rows['onset_time_s'] / REFERENCE_ONSET_TIME_BIN_S
    ).astype(int)
    onset_bin_indices = np.arange(onset_bin_count)
    onset_bin_starts_s = onset_bin_indices * REFERENCE_ONSET_TIME_BIN_S
    onset_bin_ends_s = np.minimum(onset_bin_starts_s + REFERENCE_ONSET_TIME_BIN_S, REFERENCE_ONSET_MAX_TIME_S)
    onset_bin_centers_s = (onset_bin_starts_s + onset_bin_ends_s) / 2
    onset_bin_labels = [
        f'{start_s:g}–{end_s:g}' for start_s, end_s in zip(onset_bin_starts_s, onset_bin_ends_s)
    ]
    fraction_active = pooled_rows.groupby('onset_bin')[selection['stimulus_labels']].mean().reindex(
        onset_bin_indices
    )
    neurons_per_bin = pooled_rows.groupby('onset_bin').size().reindex(onset_bin_indices, fill_value=0)
    reference_onset_recruitment[side] = {
        'reference_label': reference_label, 'neurons': pooled_rows, 'fraction_active': fraction_active,
        'onset_bin_starts_s': onset_bin_starts_s, 'onset_bin_centers_s': onset_bin_centers_s,
        'onset_bin_labels': onset_bin_labels,
        'neurons_per_bin': neurons_per_bin,
    }

    fig, (line_ax, heatmap_ax, count_ax) = plt.subplots(1, 3, figsize=(20, 4.5), constrained_layout=True)
    for stimulus_label in selection['stimulus_labels']:
        line_ax.plot(
            onset_bin_centers_s, fraction_active[stimulus_label], marker='o', linewidth=2,
            color=stimuli_colors.get(stimulus_label), label=stimulus_label,
        )
    line_ax.set(
        xlabel=f'{reference_label} onset after motion onset (s)', ylabel='Fraction of neurons active',
        title=f'{side.capitalize()}: recruitment by {REFERENCE_ONSET_TIME_BIN_S:g}-s {reference_label}-onset bins',
        xticks=onset_bin_starts_s, ylim=(0, 1),
    )
    line_ax.set_xlim(-0.05 * REFERENCE_ONSET_TIME_BIN_S, REFERENCE_ONSET_MAX_TIME_S)
    line_ax.legend(title='Stimulus', frameon=False, fontsize=8, ncol=2,
                   bbox_to_anchor=(0.5, -0.22), loc='upper center')

    heatmap_values = fraction_active.T.to_numpy(dtype=float)
    image = heatmap_ax.imshow(heatmap_values, aspect='auto', vmin=0, vmax=1, cmap='viridis')
    heatmap_ax.set(
        xlabel=f'{reference_label} onset after motion onset (s)', ylabel='Stimulus',
        title='Fraction active', xticks=np.arange(onset_bin_count),
        xticklabels=onset_bin_labels,
        yticklabels=selection['stimulus_labels'],
    )
    fig.colorbar(image, ax=heatmap_ax, label='Fraction active')

    count_ax.bar(
        onset_bin_centers_s, neurons_per_bin.to_numpy(),
        width=onset_bin_ends_s - onset_bin_starts_s, align='center', color='0.25', edgecolor='white',
    )
    count_ax.set(
        xlabel=f'{reference_label} onset after motion onset (s)', ylabel=f'# {reference_label}-active neurons',
        title=f'{reference_label} neurons per onset-time bin', xticks=onset_bin_starts_s,
    )
    count_ax.set_xlim(-0.05 * REFERENCE_ONSET_TIME_BIN_S, REFERENCE_ONSET_MAX_TIME_S)
    if SAVE_REFERENCE_ONSET_SVG:
        plot_common.save_publication_figure(
            fig, SVG_OUTPUT_DIR / f'exp05_reference_onset_recruitment_{side}', formats=('svg',), dpi=600,
        )
    plt.show()
    display(fraction_active.T)


In [ ]:
# Cell 11 - Optional reviewed-run report.
# Save this notebook before enabling REPORT_SETTINGS['save_report'].
notebook_path = repo_root / 'notebooks' / 'calcium' / 'exp_05_map_positions' / '01_all_fish_raster.ipynb'
report_result = several_reporting.save_analysis_report_run(
    settings=settings, report_settings=REPORT_SETTINGS, analysis_path=analysis_path,
    tables={'active_neuron_counts': active_neuron_counts,
            'shared_active_auc_summary': shared_active_auc_summary},
    notebook_path=notebook_path,
)
